In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'ai-forever/ruRoberta-large'
RANDOM_STATE = 42

In [ ]:
with open('data/categories.txt', 'r') as f:
    categories = f.readlines()
for i in range(len(categories)):
    categories[i] = categories[i].replace('\n', '')
map_categories = {}
for i, item in enumerate(categories):
    map_categories[item] = i
print(map_categories)

In [ ]:
marked_data = pd.read_csv('data/marked_data.csv')
generated_data = pd.read_csv('data/generated_data.csv')

marked_data = marked_data[~marked_data['category'].isin(['бытовая техника', 'электроника', 'нет категории'])]
marked_data.loc[marked_data['category'] == 'посуда', 'category'] = 'одежда'
marked_data = marked_data.rename(columns={'text': 'review'})

marked_data['source'] = ['original'] * len(marked_data)
generated_data['source'] = ['generated'] * len(generated_data)

data = pd.concat([marked_data, generated_data], axis=0)

In [ ]:
print(data['category'].value_counts())

In [ ]:
print(data['source'].value_counts())

In [ ]:
data['stratify'] = data['category'] + '_' + data['source']
train, test = train_test_split(data, test_size=0.2, random_state=RANDOM_STATE, stratify=data['stratify'])
X_train, y_train = train['review'].to_list(), train['category'].to_list()
X_test, y_test = test['review'].to_list(), test['category'].to_list()

train_dataset = Dataset.from_dict({'text': X_train, 'category': y_train})
test_dataset = Dataset.from_dict({'text': X_test, 'category': y_test})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(categories))
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir='/kaggle/working/',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    eval_steps=50,
    eval_strategy='steps',

    save_strategy='steps',
    save_steps=50,
    save_total_limit=1,

    load_best_model_at_end=True,
    metric_for_best_model='weighted_f1',
    greater_is_better=True,
    
    logging_steps=50,
    report_to='none',
)

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch['text'], padding=False)

train_dataset = train_dataset.map(tokenize_fn, batched=True).rename_column('category', 'labels').remove_columns(['text'])
test_dataset = test_dataset.map(tokenize_fn, batched=True).rename_column('category', 'labels').remove_columns(['text'])

In [ ]:
def encode_labels(batch):
    mapping = {cat: i for i, cat in enumerate(categories)}
    batch['labels'] = [mapping[label] for label in batch['labels']]
    return batch

train_dataset = train_dataset.map(encode_labels, batched=True)
test_dataset = test_dataset.map(encode_labels, batched=True)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'weighted_f1': f1_score(labels, preds, average='weighted')}

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

In [ ]:
train_loss = []
eval_loss = []
grad_norm = []
for item in trainer.state.log_history:
    if item.get('loss', None):
        train_loss.append(item['loss'])
    if item.get('eval_loss', None):
        eval_loss.append(item['eval_loss'])
    if item.get('grad_norm', None):
        grad_norm.append(item['grad_norm'])
steps = [(i + 1) * 50 for i in range(len(train_loss))]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(steps, train_loss, label='Train Loss', color='blue')
axes[0].plot(steps, eval_loss, label='Eval Loss', color='orange')
axes[0].set_title("Train Loss vs Eval Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(steps, grad_norm, label='Grad Norm', color='green')
axes[1].set_title('Gradient Norm')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Grad Norm')
axes[1].legend()

plt.tight_layout()
plt.show()